In [1]:
import pandas as pd
import sectio
from tqdm import tqdm

all_families = sectio.catalog.list_families()
results = []

for family in tqdm(all_families, desc="Auditing Families"):
    if family == "DATA_DICTIONARY": 
        continue
    try:
        df_family = sectio.catalog.get_family(family)
        table_name = f"sections_{family.lower()}"
        
        for sid in df_family['Section_ID'].tolist():
            try:
                # 1. Compute high-precision properties
                cs = sectio.db.get_section(table_name, sid, subdivision='calc')
                
                # 2. Extract DB values from metadata
                # Using 0.0 as default to ensure they are floats
                db_a  = float(cs.metadata.get('A', 0.0))
                db_iy = float(cs.metadata.get('Iy', 0.0))
                db_iz = float(cs.metadata.get('Iz', 0.0))
                db_j  = float(cs.metadata.get('It', 0.0))

                # 3. Store Raw Data
                results.append({
                    "Family": family,
                    "Section_ID": sid,
                    "DB_Area": db_a,
                    "Calc_Area": cs.area,
                    "DB_Iy": db_iy,
                    "Calc_Iy": cs.Iy,
                    "DB_Iz": db_iz,
                    "Calc_Iz": cs.Iz,
                    "DB_J": db_j,
                    "Calc_J": cs.J
                })
            except Exception:
                continue
    except Exception:
        continue

# Create the master DataFrame
audit_df = pd.DataFrame(results)

# Now you can calculate any error on the fly with Pandas:
# audit_df['Iy_Err'] = (abs(audit_df['Calc_Iy'] - audit_df['DB_Iy']) / audit_df['DB_Iy']) * 100

print(f"Audit complete! {len(audit_df)} rows captured.")
# display(audit_df.head())

Auditing Families: 100%|██████████| 22/22 [00:01<00:00, 18.17it/s]

Audit complete! 216 rows captured.


In [2]:
# 1. Calculate Absolute Differences
audit_df['Area_Diff'] = audit_df['Calc_Area'] - audit_df['DB_Area']
audit_df['Iy_Diff'] = audit_df['Calc_Iy'] - audit_df['DB_Iy']
audit_df['Iz_Diff'] = audit_df['Calc_Iz'] - audit_df['DB_Iz']

# 2. Calculate Percentage Errors (Relative to DB values)
# Using a small epsilon to avoid division by zero
epsilon = 1e-9
audit_df['Area_Err%'] = (abs(audit_df['Area_Diff']) / (audit_df['DB_Area'] + epsilon)) * 100
audit_df['Iy_Err%'] = (abs(audit_df['Iy_Diff']) / (audit_df['DB_Iy'] + epsilon)) * 100
audit_df['Iz_Err%'] = (abs(audit_df['Iz_Diff']) / (audit_df['DB_Iz'] + epsilon)) * 100

# 3. Round for readability
audit_df = audit_df.round(4)

print("✅ Error metrics calculated.")

✅ Error metrics calculated.


In [3]:
audit_df.columns

Index(['Family', 'Section_ID', 'DB_Area', 'Calc_Area', 'DB_Iy', 'Calc_Iy',
       'DB_Iz', 'Calc_Iz', 'DB_J', 'Calc_J', 'Area_Diff', 'Iy_Diff', 'Iz_Diff',
       'Area_Err%', 'Iy_Err%', 'Iz_Err%'],
      dtype='str')

In [4]:
# Create a dedicated inspection dataframe for Area
area_audit = audit_df[['Family', 'Section_ID', 'DB_Area', 'Calc_Area', 'Area_Diff', 'Area_Err%']].copy()

# Sort by the most egregious errors
area_audit = area_audit.sort_values('Area_Err%', ascending=False)

print("Top 10 Area Discrepancies:")
print(area_audit.head(10))

# Summary statistics for the 'Sanity Check'
print(f"\nMean Area Error: {area_audit['Area_Err%'].mean():.4f}%")
print(f"Max Area Error:  {area_audit['Area_Err%'].max():.4f}%")

Top 10 Area Discrepancies:
    Family   Section_ID  DB_Area   Calc_Area  Area_Diff  Area_Err%
101    CHS  CHS355.6x10  10900.0  10852.9845   -47.0155     0.4313
109    CHS  CHS406.4x10  12500.0  12448.2727   -51.7273     0.4138
161    CHS     CHS762x6  14300.0  14244.5421   -55.4579     0.3878
138    CHS   CHS610x6.3  11900.0  11943.6529    43.6529     0.3668
86     CHS  CHS273x14.2  11500.0  11540.5914    40.5914     0.3530
25     CHS    CHS88.9x4   1070.0   1066.4565    -3.5435     0.3312
162    CHS   CHS762x6.3  15000.0  14950.8340   -49.1660     0.3278
29     CHS   CHS101.6x4   1230.0   1225.9853    -4.0147     0.3264
35     CHS   CHS114.3x4   1390.0   1385.5141    -4.4859     0.3227
23     CHS    CHS76.1x5   1120.0   1116.3877    -3.6123     0.3225

Mean Area Error: 0.1056%
Max Area Error:  0.4313%


In [5]:
# 1. Filter out rows with zero database values to ensure valid percentages
active_audit = audit_df[
    (audit_df['DB_Area'] > 0) & 
    (audit_df['DB_Iy'] > 0) & 
    (audit_df['DB_Iz'] > 0)
].copy()

# 2. Define standard columns for display
cols = ['Family', 'Section_ID', 'Area_Err%', 'Iy_Err%', 'Iz_Err%']

print("🚨 TOP 10 AREA DISCREPANCIES")
print("Check for: Incorrect wall thickness, missing radii, or unit mismatches.")
print("-" * 85)
print(active_audit.sort_values('Area_Err%', ascending=False)[cols].head(10).to_string(index=False))

print("\n\n🚨 TOP 10 Iy DISCREPANCIES (Strong Axis)")
print("Check for: Flange thickness discrepancies or depth (h) mismatches.")
print("-" * 85)
print(active_audit.sort_values('Iy_Err%', ascending=False)[cols].head(10).to_string(index=False))

print("\n\n🚨 TOP 10 Iz DISCREPANCIES (Weak Axis)")
print("Check for: Taper slope (9% vs 5%) or width (b) mismatches.")
print("-" * 85)
print(active_audit.sort_values('Iz_Err%', ascending=False)[cols].head(10).to_string(index=False))

🚨 TOP 10 AREA DISCREPANCIES
Check for: Incorrect wall thickness, missing radii, or unit mismatches.
-------------------------------------------------------------------------------------
Family  Section_ID  Area_Err%  Iy_Err%  Iz_Err%
   CHS CHS355.6x10     0.4313   0.0647   0.0647
   CHS CHS406.4x10     0.4138   0.1789   0.1789
   CHS    CHS762x6     0.3878   0.2633   0.2633
   CHS  CHS610x6.3     0.3668   0.0084   0.0084
   CHS CHS273x14.2     0.3530   0.1321   0.1321
   CHS   CHS88.9x4     0.3312   0.0390   0.0390
   CHS  CHS762x6.3     0.3278   0.2883   0.2883
   CHS  CHS101.6x4     0.3264   0.1144   0.1144
   CHS  CHS114.3x4     0.3227   0.0493   0.0493
   CHS   CHS76.1x5     0.3225   0.0492   0.0492


🚨 TOP 10 Iy DISCREPANCIES (Strong Axis)
Check for: Flange thickness discrepancies or depth (h) mismatches.
-------------------------------------------------------------------------------------
Family   Section_ID  Area_Err%  Iy_Err%  Iz_Err%
   CHS CHS355.6x6.3     0.0084   0.5779   

In [6]:
# Ensure J columns exist and are numeric
active_j_audit = audit_df[
    (audit_df['DB_J'] > 0) & 
    (audit_df['Calc_J'] > 0)
].copy()

# Calculate J Error %
active_j_audit['J_Err%'] = (abs(active_j_audit['Calc_J'] - active_j_audit['DB_J']) / active_j_audit['DB_J']) * 100

cols = ['Family', 'Section_ID', 'DB_J', 'Calc_J', 'J_Err%']

print("🌪️ TOP 10 TORSIONAL CONSTANT (J) DISCREPANCIES")
print("-" * 85)
print(active_j_audit.sort_values('J_Err%', ascending=False)[cols].head(10).to_string(index=False))

print(f"\nMean J Error: {active_j_audit['J_Err%'].mean():.4f}%")

🌪️ TOP 10 TORSIONAL CONSTANT (J) DISCREPANCIES
-------------------------------------------------------------------------------------
Family  Section_ID        DB_J       Calc_J   J_Err%
   CHS CHS21.3x3.2     15400.0 1.583336e+04 2.814021
   CHS CHS21.3x2.6     13600.0 1.386062e+04 1.916356
   CHS   CHS33.7x4     83800.0 8.523090e+04 1.707519
   CHS CHS26.9x3.2     34100.0 3.465303e+04 1.621786
   CHS CHS244.5x25 210000000.0 2.128592e+08 1.361514
   CHS   CHS48.3x5    323000.0 3.270339e+05 1.248900
   CHS CHS26.9x2.6     29600.0 2.994561e+04 1.167611
   CHS CHS21.3x2.3     12600.0 1.274319e+04 1.136433
   CHS   CHS273x25 302000000.0 3.053029e+08 1.093661
   CHS CHS219.1x20 125000000.0 1.263625e+08 1.090012

Mean J Error: 0.2881%


In [7]:
audit_df.sample(1).iloc[0]

Family                CHS
Section_ID    CHS42.4x2.6
DB_Area             325.0
Calc_Area        324.9615
DB_Iy             64600.0
Calc_Iy        64592.6426
DB_Iz             64600.0
Calc_Iz        64592.6426
DB_J             129000.0
Calc_J        129710.5363
Area_Diff         -0.0385
Iy_Diff           -7.3574
Iz_Diff           -7.3574
Area_Err%          0.0119
Iy_Err%            0.0114
Iz_Err%            0.0114
Name: 9, dtype: object

In [8]:
# 1. Ensure the column names exist. 
# Check if your DB column is 'DB_J' or 'DB_It'
db_col = 'DB_It' if 'DB_It' in audit_df.columns else 'DB_J'

# 2. Filter for rows that actually have data
active_it_audit = audit_df[audit_df[db_col] > 0].copy()

# 3. Calculate Error using the new 'Calc_J' (which now uses your j_manual)
active_it_audit['J_Err%'] = (
    abs(active_it_audit['Calc_J'] - active_it_audit[db_col]) / active_it_audit[db_col]
) * 100

# 4. Display the TOP 10
cols = ['Family', 'Section_ID', db_col, 'Calc_J', 'J_Err%']
print(f"🌪️ TOP 10 TORSIONAL CONSTANT DISCREPANCIES (Using {db_col})")
print("-" * 85)
if not active_it_audit.empty:
    print(active_it_audit.sort_values('J_Err%', ascending=False)[cols].head(10).to_string(index=False))
    print(f"\nMean J Error: {active_it_audit['J_Err%'].mean():.4f}%")
else:
    print("❌ No data found! Check if 'Calc_J' or the DB column is empty.")

🌪️ TOP 10 TORSIONAL CONSTANT DISCREPANCIES (Using DB_J)
-------------------------------------------------------------------------------------
Family  Section_ID        DB_J       Calc_J   J_Err%
   CHS CHS21.3x3.2     15400.0 1.583336e+04 2.814021
   CHS CHS21.3x2.6     13600.0 1.386062e+04 1.916356
   CHS   CHS33.7x4     83800.0 8.523090e+04 1.707519
   CHS CHS26.9x3.2     34100.0 3.465303e+04 1.621786
   CHS CHS244.5x25 210000000.0 2.128592e+08 1.361514
   CHS   CHS48.3x5    323000.0 3.270339e+05 1.248900
   CHS CHS26.9x2.6     29600.0 2.994561e+04 1.167611
   CHS CHS21.3x2.3     12600.0 1.274319e+04 1.136433
   CHS   CHS273x25 302000000.0 3.053029e+08 1.093661
   CHS CHS219.1x20 125000000.0 1.263625e+08 1.090012

Mean J Error: 0.2881%


In [9]:
print(sectio.catalog.get_schema())

   column_name                                    description unit
0            h                              Height of section   mm
1            b                               Width of section   mm
2           tf                               Flange thickness   mm
3           tw                                  Web thickness   mm
4       r_root                          Radius of root fillet   mm
..         ...                                            ...  ...
60       Wz,pl  Plastic modulus of section about the z-z axis  mm3
61          Ct                     Torsional modulus constant  mm3
62       r_out                         Radius of outer fillet   mm
63        r_in                         Radius of inner fillet   mm
64           D                      Outer diameter of section   mm

[65 rows x 3 columns]


In [10]:
print(sectio.catalog.get_schema('T'))

   column_name                                        description    unit
0            h                                  Height of section      mm
1            b                                   Width of section      mm
2       r_root                              Radius of root fillet      mm
3        r_toe                        Radius of flange toe fillet      mm
4        r_web                           Radius of web toe fillet      mm
5           zs  Distance of centre of gravity to bottom along ...      mm
6          z's  Distance of centre of gravity to top along z-axis      mm
7            G                               Mass per unit length  kg.m-1
8            A                                    Area of section     mm2
9           AL                   Painting surface per unit length  m2.m-1
10          Iy           Second moment of area about the y-y axis     mm4
11          Iz           Second moment of area about the z-z axis     mm4
12         Wy1  Elastic section modulu